# 01 Data Ingest + Duplicate Handling

Notebook-first preprocessing for the comprehensive audio feature table. The key cleaning rule is applied only inside the dataframe: build a `base_call_id` by removing optional duplicate suffixes like `_02`, then keep the row with the largest `duration_sec` for each base call.

In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

AUDIO_INPUT = PROCESSED_DIR / "audio_call_feature_table_comprehensive.csv"
DEDUPED_AUDIO_OUTPUT = PROCESSED_DIR / "audio_call_feature_table_deduped.csv"
DUPLICATE_AUDIT_OUTPUT = PROCESSED_DIR / "audio_call_duplicate_audit.csv"

AUDIO_INPUT

In [ ]:
audio = pd.read_csv(AUDIO_INPUT)

profile = pd.DataFrame(
    {
        "metric": ["rows", "columns", "tickers", "call_ids", "date_min", "date_max"],
        "value": [
            len(audio),
            audio.shape[1],
            audio["ticker"].nunique(),
            audio["call_id"].nunique(),
            audio["call_date"].min(),
            audio["call_date"].max(),
        ],
    }
)
profile

In [ ]:
AUDIO_CALL_ID_RE = re.compile(r"^(?P<base>.+_\d{4}_\d{2}_\d{2})(?:_(?P<duplicate_suffix>\d{2}))?$")


def split_call_id(call_id: str) -> pd.Series:
    match = AUDIO_CALL_ID_RE.match(str(call_id))
    if match is None:
        return pd.Series({"base_call_id": str(call_id), "duplicate_suffix": pd.NA})
    return pd.Series(
        {
            "base_call_id": match.group("base"),
            "duplicate_suffix": match.group("duplicate_suffix") or pd.NA,
        }
    )

id_parts = audio["call_id"].apply(split_call_id)
audio_work = pd.concat([audio.copy(), id_parts], axis=1)
audio_work["has_duplicate_suffix"] = audio_work["duplicate_suffix"].notna()

duplicate_groups = audio_work[audio_work.duplicated("base_call_id", keep=False)].copy()
duplicate_summary = (
    duplicate_groups.sort_values(["base_call_id", "duration_sec"], ascending=[True, False])
    [["base_call_id", "call_id", "ticker", "call_date", "duration_sec", "duplicate_suffix"]]
    .reset_index(drop=True)
)

duplicate_summary

In [ ]:
# Keep the longest-duration candidate per base call. If duration ties, prefer the unsuffixed ID.
ranked = audio_work.sort_values(
    ["base_call_id", "duration_sec", "has_duplicate_suffix", "call_id"],
    ascending=[True, False, True, True],
)

deduped = ranked.drop_duplicates("base_call_id", keep="first").sort_values(
    ["ticker", "call_date", "call_id"]
)
removed = audio_work[~audio_work["call_id"].isin(deduped["call_id"])].sort_values(
    ["ticker", "call_date", "call_id"]
)

duplicate_audit = duplicate_summary.merge(
    deduped[["base_call_id", "call_id"]].rename(columns={"call_id": "kept_call_id"}),
    on="base_call_id",
    how="left",
)
duplicate_audit["decision"] = np.where(
    duplicate_audit["call_id"].eq(duplicate_audit["kept_call_id"]), "kept", "removed"
)

qa = pd.DataFrame(
    {
        "metric": [
            "input_rows",
            "duplicate_base_ids",
            "rows_in_duplicate_groups",
            "removed_rows",
            "deduped_rows",
            "remaining_duplicate_base_ids",
        ],
        "value": [
            len(audio_work),
            duplicate_groups["base_call_id"].nunique(),
            len(duplicate_groups),
            len(removed),
            len(deduped),
            int(deduped["base_call_id"].duplicated().sum()),
        ],
    }
)
qa

In [ ]:
assert deduped["base_call_id"].is_unique, "Deduped table still has duplicated base_call_id values."
assert len(deduped) + len(removed) == len(audio_work), "Dedupe accounting mismatch."

export_cols = [
    "base_call_id",
    "duplicate_suffix",
    "has_duplicate_suffix",
] + [column for column in audio.columns if column not in {"base_call_id", "duplicate_suffix", "has_duplicate_suffix"}]

deduped_export = deduped[export_cols].reset_index(drop=True)
duplicate_audit.to_csv(DUPLICATE_AUDIT_OUTPUT, index=False)
deduped_export.to_csv(DEDUPED_AUDIO_OUTPUT, index=False)

print(f"Wrote deduped audio table: {DEDUPED_AUDIO_OUTPUT}")
print(f"Wrote duplicate audit table: {DUPLICATE_AUDIT_OUTPUT}")
deduped_export.head()

In [ ]:
numeric_cols = deduped_export.select_dtypes(include="number").columns.tolist()
core_cols = [
    "duration_sec",
    "energy_mean",
    "energy_std",
    "pitch_mean",
    "pitch_std",
    "voiced_ratio",
    "audio_stress_index",
    "audio_confidence_index",
    "audio_instability_index",
    "vocal_clarity_proxy",
]
core_cols = [column for column in core_cols if column in deduped_export.columns]

summary_stats = deduped_export[core_cols].describe().T
summary_stats

In [ ]:
ticker_profile = (
    deduped_export.groupby("ticker", as_index=False)
    .agg(
        calls=("call_id", "count"),
        avg_duration_min=("duration_sec", lambda values: values.mean() / 60),
        avg_audio_stress=("audio_stress_index", "mean"),
        avg_audio_confidence=("audio_confidence_index", "mean"),
        avg_vocal_clarity=("vocal_clarity_proxy", "mean"),
    )
    .sort_values(["calls", "ticker"], ascending=[False, True])
)

ticker_profile.head(20)

In [ ]:
try:
    import matplotlib.pyplot as plt

    axes = deduped_export[core_cols].hist(figsize=(14, 10), bins=30)
    plt.suptitle("Audio feature distributions after dataframe-level dedupe", y=1.02)
    plt.tight_layout()
except ImportError:
    print("matplotlib is not installed; skipping histogram plots.")